In [5]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, TargetEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load Data (Keep ext_col and int_col for Target Encoding!)
df = pd.read_csv('used_cars.csv').drop(columns=['id'])

price_cap = df['price'].quantile(0.99)
milage_cap = df['milage'].quantile(0.99)
df_filtered = df[(df['price'] <= price_cap) & (df['milage'] <= milage_cap)].copy()

# Group-based contextual outlier removal (flags 2.66% of rows)
df_filtered['log_price'] = np.log1p(df_filtered['price'])
df_filtered['age_bucket'] = pd.cut(2024 - df_filtered['model_year'], bins=[-1, 2, 5, 10, 15, 25, 100])

q1 = df_filtered.groupby(['brand', 'age_bucket'], observed=False)['log_price'].transform(lambda x: x.quantile(0.25))
q3 = df_filtered.groupby(['brand', 'age_bucket'], observed=False)['log_price'].transform(lambda x: x.quantile(0.75))
iqr = q3 - q1

df_clean = df_filtered[
    (df_filtered['log_price'] >= (q1 - 1.5 * iqr)) & 
    (df_filtered['log_price'] <= (q3 + 1.5 * iqr))
].drop(columns=['log_price', 'age_bucket']).copy()

X = df_clean.drop(columns=['price'])
y = np.log1p(df_clean['price'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Comprehensive Feature Extraction
def extract_features_v4(X_in):
    X_out = X_in.copy()
    
    # Age & Non-linear mileage features
    X_out['vehicle_age'] = 2024 - X_out['model_year']
    X_out['miles_per_year'] = X_out['milage'] / (X_out['vehicle_age'].clip(lower=0) + 1)
    X_out['log_milage'] = np.log1p(X_out['milage'])
    
    # Richer Engine Parsing (Keep raw 'engine' string for Target Encoding!)
    eng = X_out['engine'].fillna('Missing')
    X_out['horsepower'] = eng.str.extract(r'(\d+\.?\d*)HP')[0].astype(float)
    X_out['displacement'] = eng.str.extract(r'(\d+\.?\d*)\s*L(?:iter)?', expand=False).astype(float)
    
    # Extract cylinders from both '8 Cylinder' and 'V8'/'I4'/'V6'/'W12'/'H6'
    cyl1 = eng.str.extract(r'(\d+)\s*Cylinder', expand=False).astype(float)
    cyl2 = eng.str.extract(r'\b[VIWH](\d+)\b', expand=False).astype(float)
    X_out['cylinders'] = cyl1.fillna(cyl2)
    
    X_out['hp_per_liter'] = X_out['horsepower'] / (X_out['displacement'].replace(0, np.nan))
    X_out['hp_per_cyl'] = X_out['horsepower'] / (X_out['cylinders'].replace(0, np.nan))
    X_out['is_turbo'] = eng.str.contains('Turbo|Supercharged', case=False, na=False).astype(int)
    X_out['is_electric_or_hybrid'] = (
        eng.str.contains('Electric|Hybrid|Battery', case=False, na=False) | 
        X_out['fuel_type'].fillna('').str.contains('Hybrid|Electric', case=False)
    ).astype(int)
    
    # Transmission Parsing (Keep raw 'transmission' string for Target Encoding!)
    trans = X_out['transmission'].fillna('Missing')
    X_out['trans_speeds'] = trans.str.extract(r'(\d+)[-\s]*Speed', flags=re.IGNORECASE, expand=False).astype(float)
    X_out['is_automatic'] = trans.str.contains('A/T|Automatic|Dual Shift|CVT|Auto-Shift|DCT', case=False, na=False).astype(int)
    X_out['is_manual'] = trans.str.contains('M/T|Manual', case=False, na=False).astype(int)
    
    # Preserve Missingness Signal in clean_title, accident, and fuel_type
    X_out['clean_title_flag'] = X_out['clean_title'].eq('Yes').astype(int)
    X_out['accident_clean'] = X_out['accident'].fillna('Missing')
    X_out['fuel_type_clean'] = X_out['fuel_type'].fillna('Missing')
    
    # Interaction term for Target Encoding
    X_out['brand_model'] = X_out['brand'].astype(str) + '_' + X_out['model'].astype(str)
    
    # Corrected Luxury & Exotic Brand List (matching exact dataset strings)
    luxury_list = [
        'Mercedes-Benz', 'BMW', 'Audi', 'Lexus', 'Porsche', 'Land', 'Tesla', 
        'Jaguar', 'Cadillac', 'Bentley', 'Rolls-Royce', 'Ferrari', 'Lamborghini', 
        'McLaren', 'Maserati', 'Aston', 'Rivian', 'Lucid', 'Genesis', 'Alfa', 
        'Maybach', 'Bugatti', 'Lotus'
    ]
    X_out['is_luxury'] = X_out['brand'].isin(luxury_list).astype(int)
    
    return X_out

num_features = [
    'milage', 'log_milage', 'model_year', 'vehicle_age', 'horsepower', 
    'displacement', 'cylinders', 'miles_per_year', 'hp_per_liter', 
    'hp_per_cyl', 'trans_speeds', 'clean_title_flag', 'is_turbo', 
    'is_electric_or_hybrid', 'is_automatic', 'is_manual', 'is_luxury'
]
high_cat_features = ['brand', 'model', 'brand_model', 'engine', 'transmission', 'ext_col', 'int_col']
low_cat_features = ['fuel_type_clean', 'accident_clean']

preprocessor_v4 = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median', add_indicator=True)), 
            ('scaler', StandardScaler())
        ]), num_features),
        ('high_cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')), 
            ('encoder', TargetEncoder(target_type='continuous', smooth='auto', cv=5, random_state=42))
        ]), high_cat_features),
        ('low_cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), low_cat_features)
    ]
)

# 3. Tuned HistGradientBoosting Pipeline
hgb_pipeline = Pipeline(steps=[
    ('feature_extraction', FunctionTransformer(extract_features_v4)),
    ('preprocessor', preprocessor_v4),
    ('regressor', HistGradientBoostingRegressor(
        max_iter=600,
        learning_rate=0.03,
        max_leaf_nodes=63,
        min_samples_leaf=50,
        l2_regularization=5.0,
        random_state=42
    ))
])

hgb_pipeline.fit(X_train, y_train)
y_pred_log = hgb_pipeline.predict(X_test)

c:\Users\HP\ml_projects\Project1\.venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


In [2]:
y_pred_log

array([10.75629606,  9.93937051, 11.23334505, ..., 11.08055573,
       10.83544898, 10.65499901], shape=(36974,))

In [ ]:
# Compute smearing factor from training residuals (no test leakage)
train_pred_log = hgb_pipeline.predict(X_train)
smearing_factor = np.mean(np.exp(y_train - train_pred_log)) # ~1.1055

# Apply bias-corrected back-transformation
y_pred_dollar_corrected = np.exp(y_pred_log) * smearing_factor - 1
y_test_actual = np.expm1(y_test)

print(f"Log Space R2:               {r2_score(y_test, y_pred_log):.4f}")
print(f"Dollar Space R2 (Corrected): {r2_score(y_test_actual, y_pred_dollar_corrected):.4f}")

Log Space R2:               0.7321
Dollar Space R2 (Corrected): 0.5922


: 

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Define pipelines for different models
# models = {
#     "Random Forest": Pipeline(steps=[
#         ('feature_extraction', feature_engineering_step),
#         ('preprocessor', preprocessor),
#         ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
#     ]),
#     "Gradient Boosting": Pipeline(steps=[
#         ('feature_extraction', feature_engineering_step),
#         ('preprocessor', preprocessor),
#         ('regressor', GradientBoostingRegressor(n_estimators=100, random_state=42))
#     ])
# }

# # Dictionary to store results
# results = {}

# # train and evaluate each model
# for name, pipeline, in models.items():
#     print(f"Training {name}...")
#     pipeline.fit(X_train, y_train)

#     # predict and reverse the log transformer
#     y_pred_log = pipeline.predict(X_test)
#     y_pred_actual = np.expm1(y_pred_log)
#     y_test_actual = np.expm1(y_test)

#     # calculate metrics
#     mae = mean_absolute_error(y_test_actual, y_pred_actual)
#     rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
#     r2 = r2_score(y_test_actual, y_pred_actual)

#     results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
#     print(f"{name} trained successfully.\n")

Training Random Forest...
Random Forest trained successfully.

Training Gradient Boosting...
Gradient Boosting trained successfully.



In [ ]:
# # display results as a dataframe
# results_df = pd.DataFrame(results).T

# # format output
# results_df['MAE'] = results_df['MAE'].apply(lambda x: f"${x:,.2f}")
# results_df['RMSE'] = results_df['RMSE'].apply(lambda x: f"${x:,.2f}")
# results_df['R2'] = results_df['R2'].apply(lambda x: f"{x:.4f}")

# display(results_df)

,MAE,RMSE,R2
Random Forest,"$17,623.43","$68,770.78",0.1496
Gradient Boosting,"$17,475.84","$69,172.22",0.1396


In [5]:
# Outlier removal


# 1. Remove Outliers using the 99th percentile
price_cap = df['price'].quantile(0.99)
milage_cap = df['milage'].quantile(0.99)

# Filter the dataframe to only keep cars below these caps
df_filtered = df[(df['price'] <= price_cap) & (df['milage'] <= milage_cap)].copy()

print(f"Original rows: {len(df)}")
print(f"Filtered rows: {len(df_filtered)}")
print(f"Dropped {len(df) - len(df_filtered)} extreme outliers.")

# 2. Separate and Split
X = df_filtered.drop(columns=['price'])
y = np.log1p(df_filtered['price']) 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Original rows: 188533
Filtered rows: 184866
Dropped 3667 extreme outliers.


In [ ]:
# # Hyperparam tuning with randomizedsearchcv

# from sklearn.ensemble import RandomForestRegressor
# from sklearn.model_selection import RandomizedSearchCV
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# # 1. Build the base pipeline
# rf_pipeline = Pipeline(steps=[
#     ('feature_extraction', feature_engineering_step),
#     ('preprocessor', preprocessor),
#     ('regressor', RandomForestRegressor(random_state=42, n_jobs=-1))
# ])

# # 2. Define the hyperparameter grid
# # The "regressor__" prefix tells the pipeline these settings are for the RandomForest step
# param_distributions = {
#     'regressor__n_estimators': [100, 200, 300],
#     'regressor__max_depth': [10, 15, 20, None],
#     'regressor__min_samples_split': [2, 5, 10],
#     'regressor__min_samples_leaf': [1, 2, 4]
# }

# # 3. Set up Randomized Search (tests 10 random combinations to save time)
# random_search = RandomizedSearchCV(
#     estimator=rf_pipeline,
#     param_distributions=param_distributions,
#     n_iter=10,
#     cv=3, 
#     scoring='neg_mean_absolute_error',
#     random_state=42,
#     n_jobs=-1,
#     verbose=2
# )

# print("Starting hyperparameter tuning...")
# random_search.fit(X_train, y_train)

# # 4. Extract the best model and evaluate
# best_model = random_search.best_estimator_
# print(f"\nBest Parameters Found: {random_search.best_params_}")

# # Predict on test data
# y_pred_log = best_model.predict(X_test)
# y_pred_actual = np.expm1(y_pred_log)
# y_test_actual = np.expm1(y_test)

# # Calculate final metrics
# mae = mean_absolute_error(y_test_actual, y_pred_actual)
# rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
# r2 = r2_score(y_test_actual, y_pred_actual)

# print(f"\nTuned Random Forest Metrics:")
# print(f"MAE: ${mae:,.2f}")
# print(f"RMSE: ${rmse:,.2f}")
# print(f"R-squared (R2): {r2:.4f}")

Starting hyperparameter tuning...
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best Parameters Found: {'regressor__n_estimators': 100, 'regressor__min_samples_split': 2, 'regressor__min_samples_leaf': 4, 'regressor__max_depth': 15}

Tuned Random Forest Metrics:
MAE: $13,030.60
RMSE: $25,013.14
R-squared (R2): 0.4567


In [9]:
from xgboost import XGBRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, AdaBoostRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Define pipelines with the new feature engineering step and advanced regressors
advanced_models = {
    "XGBoost": Pipeline(steps=[
        ('feature_extraction', deep_feature_engineering_step),
        ('preprocessor', preprocessor), # Ensure you run your updated ColumnTransformer
        ('regressor', XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1))
    ]),
    "HistGradientBoosting (LightGBM clone)": Pipeline(steps=[
        ('feature_extraction', deep_feature_engineering_step),
        ('preprocessor', preprocessor),
        ('regressor', HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_depth=6, random_state=42))
    ]),
    "AdaBoost": Pipeline(steps=[
        ('feature_extraction', deep_feature_engineering_step),
        ('preprocessor', preprocessor),
        ('regressor', AdaBoostRegressor(n_estimators=100, learning_rate=0.1, random_state=42))
    ])
}

# Evaluate the models
advanced_results = {}

for name, pipeline in advanced_models.items():
    print(f"Training {name}...")
    pipeline.fit(X_train, y_train)
    
    y_pred_log = pipeline.predict(X_test)
    y_pred_actual = np.expm1(y_pred_log)
    y_test_actual = np.expm1(y_test)
    
    mae = mean_absolute_error(y_test_actual, y_pred_actual)
    rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
    r2 = r2_score(y_test_actual, y_pred_actual)
    
    advanced_results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}

# Display results
adv_results_df = pd.DataFrame(advanced_results).T
adv_results_df['MAE'] = adv_results_df['MAE'].apply(lambda x: f"${x:,.2f}")
adv_results_df['RMSE'] = adv_results_df['RMSE'].apply(lambda x: f"${x:,.2f}")
adv_results_df['R2'] = adv_results_df['R2'].apply(lambda x: f"{x:.4f}")
display(adv_results_df)

Training XGBoost...
Training HistGradientBoosting (LightGBM clone)...
Training AdaBoost...


,MAE,RMSE,R2
XGBoost,"$12,899.26","$24,808.10",0.4656
HistGradientBoosting (LightGBM clone),"$12,936.78","$24,889.38",0.4621
AdaBoost,"$14,134.42","$26,694.13",0.3813


In [14]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

# 1. Define the base models (using the tuned parameters you found earlier)
base_models = [
    ('rf', RandomForestRegressor(
        n_estimators=100, 
        min_samples_split=2, 
        min_samples_leaf=4, 
        max_depth=15, 
        random_state=42, 
        n_jobs=-1
    )),
    ('xgb', XGBRegressor(
        n_estimators=300, 
        learning_rate=0.05, 
        max_depth=6, 
        subsample=0.8,
        random_state=42, 
        n_jobs=-1
    ))
]

# 2. Build the Stacking Regressor
# The Ridge regression model will learn how to optimally combine the RF and XGB predictions
stacked_model = StackingRegressor(
    estimators=base_models,
    final_estimator=Ridge(),
    cv=3,
    n_jobs=-1
)

# 3. Create the final pipeline
stacking_pipeline = Pipeline(steps=[
    ('feature_extraction', feature_engineering_v3),
    ('preprocessor', preprocessor), # Try removing colors from your preprocessor if this still stalls
    ('stacker', stacked_model)
])

# 4. Train and Evaluate
print("Training Stacking Regressor (This will take a few minutes)...")
stacking_pipeline.fit(X_train, y_train)

y_pred_log = stacking_pipeline.predict(X_test)
y_pred_actual = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

mae = mean_absolute_error(y_test_actual, y_pred_actual)
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
r2 = r2_score(y_test_actual, y_pred_actual)

print(f"\nStacked Model Metrics:")
print(f"MAE: ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R-squared (R2): {r2:.4f}")

Training Stacking Regressor (This will take a few minutes)...

Stacked Model Metrics:
MAE: $12,888.77
RMSE: $24,807.38
R-squared (R2): 0.4656


In [15]:
# Evaluate both metric spaces
y_pred_log = stacking_pipeline.predict(X_test)
y_pred_actual = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

# Dollar-Space Metrics
mae_dollar = mean_absolute_error(y_test_actual, y_pred_actual)
r2_dollar = r2_score(y_test_actual, y_pred_actual)

# Log-Space Metrics
r2_log = r2_score(y_test, y_pred_log)

print(f"MAE (Dollars): ${mae_dollar:,.2f}")
print(f"R-squared (Dollar Space): {r2_dollar:.4f}")
print(f"R-squared (Log Space):    {r2_log:.4f}")

MAE (Dollars): $12,888.77
R-squared (Dollar Space): 0.4656
R-squared (Log Space):    0.6801
